In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_dir = Path("~/Developer/project/monster-hack").expanduser()
data_dir = project_dir / "Dataset" / "Dataset"
train_dir = data_dir / "Training"
general_dir = data_dir / "General Files"

people_path = train_dir / "la_mmh_training_person_dim.csv"
lms_path = train_dir / "la_mmh_training_lms_item_activity_by_session_fact.csv"
video_path = train_dir / "la_mmh_training_video_activity_by_view_fact.csv"
calendar_path = general_dir / "la_mmh_calendar_dim.csv"
lms_dim_path = general_dir / "la_mmh_lms_item_dim.csv"
video_dim_path = general_dir / "la_mmh_video_dim.csv"
pmi_path = train_dir / "la_mmh_training_person_module_instance_fact.csv"
module_path = general_dir / "la_mmh_module_instance_dim.csv"

for path in [people_path, lms_path, video_path]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing file: {path}")


In [ ]:
# Count unique active dates separately for night and day activity.
def activity_day_counts(path, chunksize=300_000):
    night_parts = []
    day_parts = []
    for chunk in pd.read_csv(path, usecols=["P_ID", "TIMESTAMP"], dtype={"P_ID": "string", "TIMESTAMP": "string"}, chunksize=chunksize):
        chunk["TIMESTAMP"] = pd.to_datetime(chunk["TIMESTAMP"], utc=True, errors="coerce")
        chunk = chunk.dropna(subset=["P_ID", "TIMESTAMP"])
        hour = chunk["TIMESTAMP"].dt.hour + chunk["TIMESTAMP"].dt.minute / 60
        chunk["date"] = chunk["TIMESTAMP"].dt.floor("D")
        night_parts.append(chunk.loc[(hour > 0) & (hour < 5), ["P_ID", "date"]].drop_duplicates())
        day_parts.append(chunk.loc[(hour > 5) & (hour < 24), ["P_ID", "date"]].drop_duplicates())
    return pd.concat(night_parts).drop_duplicates(), pd.concat(day_parts).drop_duplicates()


In [ ]:
# Preparation complete. Run the analysis cell below for this monster type.

In [ ]:
lms_night, lms_day = activity_day_counts(lms_path)
video_night, video_day = activity_day_counts(video_path)
night_dates = pd.concat([lms_night, video_night]).drop_duplicates()
day_dates = pd.concat([lms_day, video_day]).drop_duplicates()
night_counts = night_dates.groupby("P_ID").size().rename("night_active_days")
day_counts = day_dates.groupby("P_ID").size().rename("day_active_days")
vampire_evidence = pd.concat([night_counts, day_counts], axis=1).fillna(0).reset_index()
vampire_evidence = vampire_evidence.loc[(vampire_evidence["night_active_days"] > 21) & (vampire_evidence["night_active_days"] > vampire_evidence["day_active_days"])].copy()
print(f"Found {len(vampire_evidence):,} candidate students")
display(vampire_evidence.head(20))

In [ ]:
vampire_submission = vampire_evidence[["P_ID"]].drop_duplicates().sort_values("P_ID").reset_index(drop=True)
vampire_submission.insert(0, "Monster Type", "Vampire")
display(vampire_submission)

In [ ]:
if not vampire_evidence.empty:
    display(vampire_evidence.head(1))
else:
    print("No students matched the Vampire rule.")

In [ ]:
print("Review note: The guide does not specify the timezone or how to count activity exactly at 00:00 and 05:00. This code uses UTC and excludes those boundary times.")